This first code cell mounts your Google Drive to the Colab environment. This step is essential if your project relies on files stored in your Drive, such as datasets or pre-trained models, making them accessible within the Colab runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


This cell installs the `google-generativeai` library. This library provides the necessary tools and functions to interact with Google's powerful generative AI models, like Gemini, allowing you to integrate AI capabilities into your projects.

In [ ]:
pip install google-generativeai

Similar to the previous step, this cell ensures that the `google-generativeai` library is updated to its latest version. Keeping libraries updated is good practice to access the newest features, bug fixes, and performance improvements.

In [ ]:
!pip install -U google-generativeai

This cell installs the `transformers` library from Hugging Face. This library is a popular resource for state-of-the-art natural language processing (NLP) models, including the DistilBERT model used later in this notebook for sequence classification.

In [ ]:
pip install transformers

This code block is crucial for setting up the 'judge' model. It loads a pre-trained DistilBERT tokenizer and a DistilBERT model configured for sequence classification. For this task, it's set up for regression (judging ratings) by setting `num_labels=1`. The model is then moved to the available computing device (GPU if present, otherwise CPU) and set to evaluation mode to prepare for making predictions.

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch

# Load pre-trained model and tokenizer
model_name = "distilbert-base-uncased"
loaded_tokenizer = DistilBertTokenizer.from_pretrained(model_name)
# Initialize for regression by setting num_labels=1
loaded_model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=1)

# Move model to device (GPU if available)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
loaded_model.to(device)
loaded_model.eval() # Set model to evaluation mode

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


This comprehensive code block contains the main logic for the experiment. It begins by configuring API keys and file paths. It then loads the pre-trained DistilBERT model and any existing experimental results to resume progress. Key functions are defined, including `predict_rating` (to assess generated text) and `call_gemini_smart` (for robust API calls with retry logic). The core of the cell is an experiment loop that iterates through different target ratings and prompt types ('Basic' and 'Persona'), generating text using the Gemini model and evaluating it with DistilBERT. Results are continuously saved, and finally, detailed summary statistics of the experiment's performance are generated and displayed.

In [ ]:
import pandas as pd
import numpy as np
import torch
import time
import os
import google.generativeai as genai
from tqdm import tqdm
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

# ==========================================
# 1. API and Model Configuration
# ==========================================
# Replace with your API key; update whenever switching users.
GEMINI_API_KEY = "YOUR_API_KEY_HERE"
genai.configure(api_key=GEMINI_API_KEY)

# Model to use (change to pro after you finish with flash)
MODEL_NAME = "models/gemini-3-flash-preview"

MODEL_PATH = "/content/drive/MyDrive/GenAI_Project/my_final_model"
DATA_PATH = "imdb_sup.csv"
OUTPUT_FILE = "results_gemini_3.0_flash.xlsx"
SUMMARY_FILE = "summary_statistics.xlsx"

NUM_ITERATIONS = 10
TARGET_RATINGS = [1, 2, 3, 4, 7, 8, 9, 10]

# ==========================================
# 2. Loading the Judge (DistilBERT) and existing data
# ==========================================
print("Loading Judge Model and existing data...")
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
loaded_model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
loaded_tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)

def load_existing_results(file_path):
    if os.path.exists(file_path):
        try:
            return pd.read_excel(file_path).to_dict('records')
        except:
            return []
    return []

results = load_existing_results(OUTPUT_FILE)
print(f"Found {len(results)} existing records. Resuming from where we stopped...")

def predict_rating(text):
    if not text or text == "ERROR": return 0, 0
    inputs = loaded_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = loaded_model(**inputs)
    logits = outputs.logits
    if logits.shape[1] > 1: # Classifier
        pred_label = torch.argmax(logits, dim=1).item()
        label_to_rating = {0:1, 1:2, 2:3, 3:4, 4:7, 5:8, 6:9, 7:10}
        return float(pred_label), label_to_rating.get(pred_label, 1)
    else: # Regression
        raw_val = logits.item()
        valid = np.array([1, 2, 3, 4, 7, 8, 9, 10])
        return raw_val, valid[(np.abs(valid - raw_val)).argmin()]

# ==========================================
# 3. Smart Call Function (2 Retries and Stop)
# ==========================================
def call_gemini_smart(prompt):
    model = genai.GenerativeModel(MODEL_NAME)
    retry_count = 0
    while True:
        try:
            response = model.generate_content(prompt, request_options={"timeout": 60})
            return response.text.strip(), "SUCCESS"
        except Exception as e:
            err_msg = str(e).lower()
            if "429" in err_msg:
                retry_count += 1
                if retry_count >= 2: # Per request: only 2 attempts
                    print(f"\n🛑 DAILY LIMIT (RPD) LIKELY REACHED. Failed 2 attempts. Stopping...")
                    return None, "DAILY_LIMIT"

                print(f"\n⚠️ Rate Limit (RPM). Waiting 60s (Attempt {retry_count}/2)...")
            else:
                print(f"\n❌ API Error: {e}. Retrying in 10s...")


# ==========================================
# 4. Experiment Loop
# ==========================================
df = pd.read_csv(DATA_PATH)
stop_execution = False

print(f"Starting/Resuming Experiment for: {MODEL_NAME}")

for rating in TARGET_RATINGS:
    if stop_execution: break
    rating_df = df[df['Rating'] == rating]

    for p_type in ["Basic", "Persona"]:
        if stop_execution: break

        # Checks how many we've already done to skip if necessary
        done_count = sum(1 for r in results if r['Target_Rating'] == rating and r['Prompt_Type'] == p_type)
        needed = NUM_ITERATIONS - done_count

        if needed <= 0: continue

        print(f"\n>>> Rating {rating} | {p_type} | Progress: {done_count}/{NUM_ITERATIONS}")

        for i in tqdm(range(needed)):
            sampled = rating_df.sample(n=10, replace=True)
            reviews_block = "\n---\n".join(sampled['Review'].astype(str).tolist())

            if p_type == "Basic":
                prompt = f"Based on these 10 reviews (all rated {rating}/10), write one short representative review. No headers.\n\nReviews:\n{reviews_block}"
            else:
                persona = "an angry critic" if rating <= 4 else "a happy fan"
                prompt = f"You are {persona}. Write a short review based on these 10 reviews (all rated {rating}/10). No headers.\n\nReviews:\n{reviews_block}"

            gen_text, status = call_gemini_smart(prompt)

            if status == "DAILY_LIMIT":
                stop_execution = True
                break

            raw, final = predict_rating(gen_text)
            results.append({
                "Target_Rating": rating,
                "Model": "2.5 Flash",
                "Prompt_Type": p_type,
                "Generated_Text": gen_text,
                "DistilBERT_Rating": final,
                "Error": abs(rating - final)
            })

            # Continuous saving for backup
            pd.DataFrame(results).to_excel(OUTPUT_FILE, index=False)
            time.sleep(2) # Slight delay to prevent overload

# ==========================================
# 5. Generating detailed statistics and saving
# ==========================================
results_df = pd.DataFrame(results)

if not results_df.empty:
    summary_df = results_df.groupby(['Model', 'Prompt_Type', 'Target_Rating']).agg(
        Mean_Error=('Error', 'mean'),
        Accuracy_Pct=('Error', lambda x: (x == 0).mean() * 100),
        Std_Deviation=('Error', 'std'),
        Sample_Count=('Generated_Text', 'count')
    ).reset_index()

    summary_df.to_excel(SUMMARY_FILE, index=False)
    results_df.to_excel(OUTPUT_FILE, index=False)

    print(f"\n✅ Process paused/completed. Total samples in Excel: {len(results_df)}")
    print("\n--- Summary of performance so far ---")
    print(summary_df)
else:
    print("\n❌ No data found for processing.")

Loading Judge Model and existing data...
Found 152 existing records. Resuming from where we stopped...
Starting/Resuming Experiment for: models/gemini-3-flash-preview

>>> Rating 10 | Persona | Progress: 2/10


100%|██████████| 8/8 [01:22<00:00, 10.30s/it]


✅ התהליך הושהה/הסתיים. סה"כ דגימות באקסל: 160

--- סיכום ביצועים עד כה ---
        Model Prompt_Type  Target_Rating  Mean_Error  Accuracy_Pct  \
0   2.5 Flash       Basic              1         0.0         100.0   
1   2.5 Flash       Basic              2         0.3          70.0   
2   2.5 Flash       Basic              3         0.4          60.0   
3   2.5 Flash       Basic              4         0.2          80.0   
4   2.5 Flash       Basic              7         1.1           0.0   
5   2.5 Flash       Basic              8         1.1           0.0   
6   2.5 Flash       Basic              9         0.7          30.0   
7   2.5 Flash       Basic             10         0.1          90.0   
8   2.5 Flash     Persona              1         0.1          90.0   
9   2.5 Flash     Persona              2         0.8          20.0   
10  2.5 Flash     Persona              3         0.8          40.0   
11  2.5 Flash     Persona              4         1.5          40.0   
12  2.5 Flash 